In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.datasets import wage_panel

from pymargins import GComputation  # 0.4.0: Margins -> GComputation

df = wage_panel.load().reset_index()
df["educ_band"] = pd.cut(
    df["educ"], [0, 11, 12, 20], labels=["<HS", "HS", "college"]
).astype(str)
print(df[["nr", "year", "lwage", "union", "married", "exper", "educ",
          "educ_band"]].head())
print("\nObservations per education band:")
print(df["educ_band"].value_counts())

   nr  year     lwage  union  married  exper  educ educ_band
0  13  1980  1.197540      0        0      1    14   college
1  13  1981  1.853060      1        0      2    14   college
2  13  1982  1.344462      0        0      3    14   college
3  13  1983  1.433213      0        0      4    14   college
4  13  1984  1.568125      0        0      5    14   college

Observations per education band:
educ_band
HS         1848
<HS        1472
college    1040
Name: count, dtype: int64


In [2]:
fit = smf.ols(
    "lwage ~ union * educ_band + exper * educ_band + expersq + married + C(year)",
    data=df,
).fit()
m = GComputation(fit,
    vcov={"type": "cluster", "groups": df["nr"]},  # cluster by worker
    at="overall", scale="identity")

In [3]:
profiles = m.predict(atexog={"union": [0, 1]}, over="educ_band")
print(profiles.summary())

                        Graph Result (delta, level=0.95)                       
                            estimate  std err        z  P>|z|  [95% Conf. Int.]
-------------------------------------------------------------------------------
educ_band=<HS, union=0        1.4286   0.0299  47.8568  0.000    1.3701, 1.4871
educ_band=<HS, union=1        1.6163   0.0484  33.3711  0.000    1.5214, 1.7113
educ_band=HS, union=0         1.6158   0.0240  67.4288  0.000    1.5688, 1.6627
educ_band=HS, union=1         1.8381   0.0328  55.9693  0.000    1.7737, 1.9025
educ_band=college, union=0    1.8210   0.0355  51.3477  0.000    1.7515, 1.8905
educ_band=college, union=1    1.9201   0.0683  28.1324  0.000    1.7863, 2.0538

n = 4360
Delta-vs-sim disagreement: 0.227%
plan e01cd0c@1 | κ = max 0.000


In [4]:
bands = ["<HS", "HS", "college"]
scenarios = []
for b in bands:
    scenarios.append({"atexog": {"union": 1, "educ_band": b}, "label": f"{b}:union"})
    scenarios.append({"atexog": {"union": 0, "educ_band": b}, "label": f"{b}:nonunion"})

# premium[b] = (union cell) − (non-union cell) for band b
weights = {}
for i, b in enumerate(bands):
    w = [0] * len(scenarios)
    w[2 * i] = 1
    w[2 * i + 1] = -1
    weights[f"premium[{b}]"] = w

# add a difference-of-premiums so we can read its CI directly
w = [0] * len(scenarios)
w[0], w[1] = -1, 1          # subtract <HS premium
w[4], w[5] = 1, -1          # add college premium
weights["college − <HS"] = w

premiums = m.contrasts(scenarios=scenarios, contrasts=weights)
print(premiums.summary())

                   Graph Result (delta, level=0.95)                  
                  estimate  std err        z  P>|z|  [95% Conf. Int.]
---------------------------------------------------------------------
premium[<HS]        0.1877   0.0529   3.5508  0.000    0.0841, 0.2913
premium[HS]         0.2223   0.0373   5.9548  0.000    0.1492, 0.2955
premium[college]    0.0991   0.0712   1.3922  0.164   -0.0404, 0.2386
college − <HS      -0.0886   0.0885  -1.0014  0.317   -0.2620, 0.0848

n = 4360
Delta-vs-sim disagreement: 11.964%
plan e01cd0c@1 | κ = max 0.000


In [5]:
ret = m.dydx("exper", over="educ_band")
print(ret.summary())

                       Graph Result (delta, level=0.95)                      
                          estimate  std err        z  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------------------------
educ_band=<HS, exper        0.0543   0.0082   6.6563  0.000    0.0383, 0.0703
educ_band=HS, exper         0.0363   0.0012  29.9430  0.000    0.0339, 0.0387
educ_band=college, exper    0.0670   0.0210   3.1986  0.001    0.0260, 0.1081

n = 4360
Delta-vs-sim disagreement: 93.925%
plan e01cd0c@1 | κ = max 0.000
